# 04D1C_v5 — LANDFIRE Feature Architecture Freeze

**Build ID: `04D1C_V5_LANDFIRE_FEATURE_ARCHITECTURE`**

04D0 froze the retrospective LANDFIRE vintage mapping.

04D1A validated the raw ArcGIS ImageServer extraction semantics.

04D1B independently confirmed LF2023 FBFM40 against:

- LANDFIRE GeoServer WCS; and
- the publisher-distributed Full Extent artifact with matching MD5/ZIP CRC.

This notebook does **not** launch full LANDFIRE production.

Its purpose is to freeze the feature architecture before production:

1. define the exact spatial regions sampled relative to each fireline unit;
2. define FBFM40 categorical features without averaging class codes;
3. define canopy summaries while respecting NoData, valid zero, class-midpoint semantics, and censored top bins;
4. define directional missingness rules;
5. separate **core model**, **extended model**, and **QA/provenance-only** fields;
6. design the scalable 10 km block extraction architecture;
7. quantify the production workload;
8. write machine-readable feature and region specifications for 04D1D.

---

## Scientific interpretation of LANDFIRE at the suppression line

LANDFIRE is a **pre-event landscape product**.

Therefore values sampled at the fireline location represent:

> the underlying/pre-event fuel and canopy environment where the suppression line was placed,

not the post-construction fuel condition or actual cleared line width.

The 30 m LANDFIRE grid generally cannot resolve a narrow constructed suppression line.

This distinction must be retained in the manuscript and feature names.

---

## Direction convention

04A defines the approach vector from the observed fire perimeter **toward the fireline**.

Therefore, for a line sample point \(p\) and unit approach direction \(\hat a\):

\[
p_{\text{fireward}}(d)=p-d\hat a
\]

moves back toward the incoming fire, while

\[
p_{\text{protected}}(d)=p+d\hat a
\]

continues beyond the line.

Directional context is generated **only** when the frozen 04A approach direction is reliable.

No target labels are used to orient or select context samples.

## V5 corrections

V1 incorrectly required stored `approach_vector_dx_m` and
`approach_vector_dy_m` fields from the 04A table.

Those stored vector components are not used by the LANDFIRE feature geometry.
The directional offsets are generated directly from the frozen
`approach_azimuth_deg`:

\[
\hat a_E=\sin(\theta), \qquad
\hat a_N=\cos(\theta).
\]

Therefore V2 requires only the azimuth, the frozen direction-reliability flag,
and the approach-vs-line angle. No 04A dataset is modified.


## V3 FBFM40 domain correction

V2 incorrectly asserted that the LANDFIRE FBFM40 raster contained 47
valid non-NoData numeric codes.

The group definitions themselves were correct:

- NB: 5 nonburnable codes
- GR: 9
- GS: 4
- SH: 9
- TU: 5
- TL: 9
- SB: 4

Thus:

\[
5 + 9 + 4 + 9 + 5 + 9 + 4 = 45.
\]

The 40 Scott–Burgan **burnable** models are the GR/GS/SH/TU/TL/SB
classes; LANDFIRE additionally encodes 5 nonburnable classes
(NB1, NB2, NB3, NB8, NB9).

V3 therefore freezes:

- 40 burnable fuel-model codes;
- 5 nonburnable codes;
- 45 valid non-NoData FBFM40 raster codes total.

This changes only the erroneous domain-count assertion and the resulting
extended-feature dimensionality. It does not alter any actual code values,
group mappings, spatial geometry, or extraction semantics.



## V4 scientific refinements

V3 passed all architecture checks. V4 refines the feature contract before
production.

### 1. Preserve line-length exposure; do not switch to area swaths

The 03L sample trajectory is retained because the desired estimand is
**line-length-weighted environmental exposure along the suppression unit**.

However, 25 m line samples can map repeatedly to the same 30 m LANDFIRE cell.
Therefore 04D1D must, after raster indexing:

1. identify raster `(row, col)` for every context request;
2. collapse duplicate cells within `unit_uid × region × product grid`;
3. **sum** their `context_weight_m`;
4. compute categorical/continuous summaries on those unique cells.

This removes duplicate-row artifacts without changing the physical
line-length weighting.

### 2. Fuel-side contrasts

Core directional features include explicit fireward-minus-protected contrasts
and Jensen-Shannon divergence of broad FBFM40 compositions.

### 3. Nonburnable subclasses

The single NB broad group remains in the 7-group composition, but core features
also retain the five individual LANDFIRE nonburnable classes:

- NB1 / 91
- NB2 / 92
- NB3 / 93
- NB8 / 98
- NB9 / 99

This avoids assuming that all nonburnable classes have the same operational
meaning.

### 4. Canopy structural conditioning

Canopy summaries separate:

- valid zero/nonforest fraction;
- unconditional landscape mean;
- positive/forest-structure conditional statistics.

### 5. Derived crown structure

Where CH, CBH and CBD are jointly valid, compute per-cell:

\[
D_c = \max(CH-CBH,0)
\]

and

\[
CFL = CBD \cdot D_c.
\]

Also retain the fraction of structurally inconsistent cells with `CBH > CH`.

### 6. Weakest-point summaries

Context samples preserve their originating line position. 04D1D must first
compute selected fireward quantities per line position, then take robust
arc-length-weighted upper-tail summaries across positions. The **p90** is core;
the raw maximum is extended.

### 7. Angle of attack

`approach_vs_line_angle_deg` is already a frozen non-LANDFIRE geometry feature.
It should be included in the integrated model feature set rather than
duplicated as a LANDFIRE feature.

### 8. Long-range spotting context

A 500–1000 m band is **not** added to the core architecture in V4. It remains
an optional future ablation because adding it would change the production halo
and significantly broaden the physical question beyond immediate line
environment.


## V5 QA correction

V4 correctly removed `burnable_fraction` and `nonburnable_fraction` from
`aggregate_fbfm40`, because:

\[
\text{nonburnable fraction} = P(NB)
\]

and

\[
\text{burnable fraction} = 1-P(NB).
\]

The V4 synthetic QA cell still referenced the deleted
`burnable_fraction` key, producing a `KeyError`.

V5 fixes only that stale test. It now verifies:

- the redundant keys are absent;
- the NB1 subclass fraction is correct;
- the seven broad FBFM40 group fractions sum to one.

No feature architecture, aggregation semantics, or production geometry changes.


## 0. Configuration

In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "04D1C_V5_LANDFIRE_FEATURE_ARCHITECTURE"

from pathlib import Path
import sys, subprocess, importlib.util

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))

FROZEN_COHORT_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "fireline_corrected_local_250m_10m" /
    "frozen_cohort" /
    "fireline_ml_dynamic_timing_cohort_frozen_v1.parquet"
)

APPROACH_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "fireline_extended_exact_approach_v3" /
    "extended_exact_approach_features_v3.parquet"
)

TERRAIN_UNIT_INDEX_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "terrain_3dep_production_v1" /
    "terrain_operational_unit_index_v1.parquet"
)

TERRAIN_BLOCK_SAMPLE_ROOT = (
    PROJECT_ROOT / "data" / "processed" /
    "terrain_3dep_production_v1" /
    "block_samples"
)

OUTPUT_ROOT = (
    PROJECT_ROOT / "outputs" /
    "landfire_feature_architecture_v5"
)

PROCESSED_ROOT = (
    PROJECT_ROOT / "data" / "processed" /
    "landfire_feature_architecture_v5"
)

for p in [OUTPUT_ROOT, PROCESSED_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

for p in [
    FROZEN_COHORT_PATH,
    APPROACH_PATH,
    TERRAIN_UNIT_INDEX_PATH,
    TERRAIN_BLOCK_SAMPLE_ROOT,
]:
    assert p.exists(), p

# ------------------------------------------------------------
# Frozen temporal mapping.
# ------------------------------------------------------------

VERSION_BY_FIRE_YEAR = {
    2020: "LF2016R",
    2021: "LF2019L",
    2022: "LF2020",
    2023: "LF2022",
    2024: "LF2023",
}

SOURCE_STATUS = {
    "LF2016R": "resolved",
    "LF2019L": "archive_pending",
    "LF2020": "archive_pending",
    "LF2022": "resolved",
    "LF2023": "resolved",
}

PRODUCTS = ["FBFM40", "CC", "CH", "CBH", "CBD"]

# ------------------------------------------------------------
# Spatial feature architecture.
#
# 03L line samples are ~25 m apart and carry arc-length weights.
#
# Directional offsets are deliberately compact:
#   near = immediate 1–3-cell neighborhood;
#   far  = broader incoming/beyond-line environment.
#
# They are discrete trajectory-strip samples, not polygon area means.
# ------------------------------------------------------------

REGION_DEPTHS_M = {
    "line": [0.0],
    "fireward_near": [-30.0, -60.0, -90.0],
    "fireward_far": [-150.0, -210.0, -300.0],
    "protected_near": [30.0, 60.0, 90.0],
    "protected_far": [150.0, 210.0, 300.0],
}

DIRECTIONAL_REGIONS = {
    "fireward_near",
    "fireward_far",
    "protected_near",
    "protected_far",
}

MAX_CONTEXT_OFFSET_M = max(
    abs(d)
    for depths in REGION_DEPTHS_M.values()
    for d in depths
)

# Existing terrain blocks are 10 km.
BLOCK_SIZE_M = 10_000.0

# 300 m maximum context + two native LANDFIRE cells of safety.
LANDFIRE_BLOCK_HALO_M = MAX_CONTEXT_OFFSET_M + 60.0

# Model tiers.
CORE_TIER = "core"
EXTENDED_TIER = "extended"
QA_TIER = "qa_only"

print("=" * 100)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 100)
print("Maximum directional context:", MAX_CONTEXT_OFFSET_M, "m")
print("Production block halo:", LANDFIRE_BLOCK_HALO_M, "m")

## 1. Imports

In [ ]:
required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "pyarrow": "pyarrow",
}

for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", package]
        )

from datetime import datetime, timezone
import json
import math
import os

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 500)
pd.set_option("display.width", 360)

## 2. Load the frozen operational unit universe

In [ ]:
frozen = pd.read_parquet(FROZEN_COHORT_PATH)
approach = pd.read_parquet(APPROACH_PATH)
unit_index = pd.read_parquet(TERRAIN_UNIT_INDEX_PATH)

assert frozen["unit_uid"].is_unique
assert approach["unit_uid"].is_unique
assert unit_index["unit_uid"].is_unique

print("Frozen cohort:", f"{len(frozen):,}")
print("Approach rows:", f"{len(approach):,}")
print("Terrain operational units:", f"{len(unit_index):,}")

# 04C1 operational universe is the clean-anchor feature universe.
base = unit_index.copy()

required_base = [
    "unit_uid",
    "_irwin_norm",
    "IncidentName",
    "Year",
    "_engagement_norm",
    "terrain_centroid_x_m",
    "terrain_centroid_y_m",
    "terrain_block_id",
    "terrain_block_ix",
    "terrain_block_iy",
]

missing = [
    c for c in required_base
    if c not in base.columns
]

if missing:
    raise RuntimeError(
        f"Missing terrain unit-index columns: {missing}"
    )

# Only columns actually required by the LANDFIRE feature geometry.
# Stored dx/dy vector columns are not required because the unit vector
# is derived directly from approach_azimuth_deg later in this notebook.
approach_columns = [
    "unit_uid",
    "approach_azimuth_deg",
    "approach_direction_reliable",
    "approach_vs_line_angle_deg",
]

missing_approach = [
    c for c in approach_columns
    if c not in approach.columns
]

if missing_approach:
    raise RuntimeError(
        f"Missing 04A directional columns: {missing_approach}"
    )

print("04A directional columns used by 04D1C:", approach_columns)
print("04A SCHEMA FOR LANDFIRE ARCHITECTURE: PASS")

# Avoid duplicate direction columns if 04C1 already carried them.
drop_from_base = [
    c for c in approach_columns
    if c != "unit_uid" and c in base.columns
]

base = base.drop(
    columns=drop_from_base,
    errors="ignore",
)

base = base.merge(
    approach[approach_columns],
    on="unit_uid",
    how="left",
    validate="one_to_one",
)

base["Year"] = pd.to_numeric(
    base["Year"],
    errors="raise",
).astype(int)

base["landfire_version_key"] = (
    base["Year"].map(VERSION_BY_FIRE_YEAR)
)

if base["landfire_version_key"].isna().any():
    years = sorted(
        base.loc[
            base["landfire_version_key"].isna(),
            "Year",
        ].unique()
    )
    raise RuntimeError(
        f"Unmapped fire years: {years}"
    )

base["landfire_source_status"] = (
    base["landfire_version_key"].map(SOURCE_STATUS)
)

base["landfire_directional_context_valid"] = (
    base["approach_direction_reliable"]
    .fillna(False)
    & pd.to_numeric(
        base["approach_azimuth_deg"],
        errors="coerce",
    ).notna()
)

assert len(base) == len(unit_index)
assert base["unit_uid"].is_unique

print("Operational LANDFIRE units:", f"{len(base):,}")
print(
    "Directional context valid:",
    f"{int(base['landfire_directional_context_valid'].sum()):,}",
    f"({base['landfire_directional_context_valid'].mean()*100:.3f}%)",
)

## 3. Frozen vintage/source-status audit

In [ ]:
vintage_summary = (
    base.groupby(
        [
            "Year",
            "landfire_version_key",
            "landfire_source_status",
        ],
        as_index=False,
    )
    .agg(
        units=("unit_uid", "size"),
        incidents=("_irwin_norm", "nunique"),
        directional_valid_units=(
            "landfire_directional_context_valid",
            "sum",
        ),
    )
)

vintage_summary["directional_valid_pct"] = (
    vintage_summary["directional_valid_units"]
    / vintage_summary["units"]
    * 100.0
)

display(vintage_summary)

vintage_summary.to_csv(
    OUTPUT_ROOT / "LANDFIRE_VINTAGE_SOURCE_STATUS_V5.csv",
    index=False,
)

assert set(
    vintage_summary["landfire_version_key"]
) == set(VERSION_BY_FIRE_YEAR.values())

print("FROZEN TEMPORAL MAPPING UNCHANGED: PASS")


## 3B. Direction-availability outcome association — QA only

Directional-feature availability may correlate with incident complexity,
observation cadence, fire size, or outcome.

This table is diagnostic only. The outcome is **not** used to construct any
LANDFIRE sampling region.

A strong association must later be handled through:

- reporting;
- direction-available sensitivity analysis; and
- comparison with a nondirectional feature model.


In [ ]:

direction_outcome_qa = (
    base.groupby(
        [
            "_engagement_norm",
            "landfire_directional_context_valid",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        units=("unit_uid", "size"),
        incidents=("_irwin_norm", "nunique"),
    )
)

direction_outcome_qa["within_outcome_pct"] = (
    direction_outcome_qa["units"]
    / direction_outcome_qa.groupby(
        "_engagement_norm"
    )["units"].transform("sum")
    * 100.0
)

display(direction_outcome_qa)

direction_outcome_qa.to_csv(
    OUTPUT_ROOT / "LANDFIRE_DIRECTION_AVAILABILITY_OUTCOME_QA_V5.csv",
    index=False,
)


## 4. Region definitions

In [ ]:
region_rows = []

for region, depths in REGION_DEPTHS_M.items():
    direction = (
        "none"
        if region == "line"
        else (
            "toward_observed_incoming_fire"
            if region.startswith("fireward")
            else "beyond_line_along_approach_vector"
        )
    )

    region_rows.append({
        "region": region,
        "directional": region in DIRECTIONAL_REGIONS,
        "depths_m": json.dumps(depths),
        "depth_sample_count": len(depths),
        "minimum_distance_m": min(abs(d) for d in depths),
        "maximum_distance_m": max(abs(d) for d in depths),
        "direction_semantics": direction,
        "sample_weight_rule": (
            "03L arc-length sample_weight_m"
            if region == "line"
            else (
                "03L arc-length sample_weight_m divided by "
                "number of depths in region"
            )
        ),
    })

region_spec = pd.DataFrame(region_rows)

display(region_spec)

region_spec.to_csv(
    OUTPUT_ROOT / "LANDFIRE_REGION_DEFINITION_V5.csv",
    index=False,
)

## 5. Directional point construction

04A azimuth is clockwise from north.

For a unit approach azimuth \(	heta\):

\[
\hat a_E=\sin	heta,\qquad
\hat a_N=\cos	heta.
\]

Negative offsets move toward the observed incoming fire.

Positive offsets continue through/beyond the line.

The function below is the exact geometry contract intended for 04D1D.

In [ ]:
def build_context_points(
    line_samples,
    unit_direction,
):
    """
    Expand weighted 03L line samples into line-relative LANDFIRE
    sampling locations.

    Inputs
    ------
    line_samples:
        DataFrame with unit_uid, x_m, y_m, sample_weight_m.
    unit_direction:
        DataFrame with unit_uid, approach_azimuth_deg,
        landfire_directional_context_valid.

    Returns
    -------
    DataFrame with one row per requested raster-sampling location.

    Important:
    - label/engagement is never used;
    - unreliable directional regions are not generated;
    - line samples always remain available.
    """

    x = line_samples.merge(
        unit_direction[
            [
                "unit_uid",
                "approach_azimuth_deg",
                "landfire_directional_context_valid",
            ]
        ],
        on="unit_uid",
        how="left",
        validate="many_to_one",
    )

    # Preserve the originating along-line position for weakest-point features.
    # A deterministic within-unit position index is sufficient; no target
    # information enters this identifier.
    x = x.copy()
    x["line_position_index"] = (
        x.groupby("unit_uid", sort=False).cumcount()
    )

    theta = np.deg2rad(
        pd.to_numeric(
            x["approach_azimuth_deg"],
            errors="coerce",
        ).to_numpy(float)
    )

    east = np.sin(theta)
    north = np.cos(theta)

    frames = []

    # Underlying pre-event fuel at the line location.
    line = x[
        [
            "unit_uid",
            "x_m",
            "y_m",
            "sample_weight_m",
            "line_position_index",
        ]
    ].copy()

    line["region"] = "line"
    line["offset_m"] = 0.0
    line["context_weight_m"] = (
        line["sample_weight_m"]
    )

    frames.append(line)

    valid = (
        x["landfire_directional_context_valid"]
        .fillna(False)
        .to_numpy(bool)
    )

    for region in sorted(DIRECTIONAL_REGIONS):
        depths = REGION_DEPTHS_M[region]

        for offset in depths:
            g = x.loc[
                valid,
                [
                    "unit_uid",
                    "x_m",
                    "y_m",
                    "sample_weight_m",
                    "line_position_index",
                ],
            ].copy()

            idx = np.flatnonzero(valid)

            g["x_m"] = (
                g["x_m"].to_numpy(float)
                + float(offset) * east[idx]
            )

            g["y_m"] = (
                g["y_m"].to_numpy(float)
                + float(offset) * north[idx]
            )

            g["region"] = region
            g["offset_m"] = float(offset)

            g["context_weight_m"] = (
                g["sample_weight_m"]
                / len(depths)
            )

            frames.append(g)

    result = pd.concat(
        frames,
        ignore_index=True,
    )

    return result


# Synthetic convention test.
synthetic_line = pd.DataFrame({
    "unit_uid": ["u1"],
    "x_m": [1000.0],
    "y_m": [2000.0],
    "sample_weight_m": [25.0],
})

# Azimuth 90° points east: fire -> line.
synthetic_direction = pd.DataFrame({
    "unit_uid": ["u1"],
    "approach_azimuth_deg": [90.0],
    "landfire_directional_context_valid": [True],
})

synthetic = build_context_points(
    synthetic_line,
    synthetic_direction,
)

display(synthetic)

# Fireward must move west for an eastward fire->line vector.
assert (
    synthetic.loc[
        synthetic["region"] == "fireward_near",
        "x_m",
    ].max()
    < 1000.0
)

# Protected must move east.
assert (
    synthetic.loc[
        synthetic["region"] == "protected_near",
        "x_m",
    ].min()
    > 1000.0
)

print("DIRECTION CONVENTION TEST: PASS")


## 5B. Native-grid duplicate-cell collapse contract

Raster-cell identity is only known after a product tile is opened and context
coordinates are converted with that tile's returned transform.

For every product tile, 04D1D must collapse repeated native cells before
feature aggregation.

The weight attached to a unique raster cell is the **sum of the arc-length
weights of all context requests that hit that cell**.

This is intentional: a cell intersected by more of the represented line length
should contribute more, but it must appear as one categorical observation for
richness/count QA.


In [ ]:

def collapse_native_cell_hits(sampled):
    """
    Production contract after raster indexing.

    Required columns:
      unit_uid, region, line_position_index,
      raster_row, raster_col, context_weight_m, raw_value

    Returns one row per unique native raster cell within unit × region,
    with line-length weights summed.
    """
    required = {
        "unit_uid",
        "region",
        "line_position_index",
        "raster_row",
        "raster_col",
        "context_weight_m",
        "raw_value",
    }

    missing = sorted(required - set(sampled.columns))

    if missing:
        raise RuntimeError(
            f"Missing native-cell collapse columns: {missing}"
        )

    x = sampled.copy()

    # One cell can be reached from several adjacent 03L points/depth requests.
    # For region-level summaries, collapse by native cell and sum represented
    # line-length weight.
    collapsed = (
        x.groupby(
            [
                "unit_uid",
                "region",
                "raster_row",
                "raster_col",
            ],
            as_index=False,
            sort=False,
        )
        .agg(
            raw_value=("raw_value", "first"),
            context_weight_m=("context_weight_m", "sum"),
            contributing_requests=("raw_value", "size"),
            contributing_positions=("line_position_index", "nunique"),
        )
    )

    # Defensive consistency: a native cell cannot have two different raw
    # categorical/numeric values in one product tile.
    value_nunique = (
        x.groupby(
            [
                "unit_uid",
                "region",
                "raster_row",
                "raster_col",
            ],
            sort=False,
        )["raw_value"]
        .nunique(dropna=False)
    )

    if (value_nunique > 1).any():
        raise RuntimeError(
            "A native raster cell produced multiple raw values."
        )

    return collapsed


# Synthetic QA: duplicated cell must become one row with summed weight.
_duplicate_test = pd.DataFrame({
    "unit_uid": ["u", "u", "u"],
    "region": ["line", "line", "line"],
    "line_position_index": [0, 1, 2],
    "raster_row": [10, 10, 11],
    "raster_col": [20, 20, 20],
    "context_weight_m": [10.0, 15.0, 20.0],
    "raw_value": [141, 141, 142],
})

_duplicate_out = collapse_native_cell_hits(_duplicate_test)

assert len(_duplicate_out) == 2
assert math.isclose(
    _duplicate_out.loc[
        (_duplicate_out["raster_row"] == 10)
        & (_duplicate_out["raster_col"] == 20),
        "context_weight_m",
    ].iloc[0],
    25.0,
)

print("NATIVE-CELL COLLAPSE CONTRACT: PASS")


## 6. FBFM40 semantics and group mapping

In [ ]:
FBFM40_GROUP_CODES = {
    "NB": {91, 92, 93, 98, 99},
    "GR": set(range(101, 110)),
    "GS": set(range(121, 125)),
    "SH": set(range(141, 150)),
    "TU": set(range(161, 166)),
    "TL": set(range(181, 190)),
    "SB": set(range(201, 205)),
}

FBFM40_VALID_CODES = set().union(
    *FBFM40_GROUP_CODES.values()
)

# LANDFIRE FBFM40 raster domain:
#   40 Scott-Burgan burnable models
#   + 5 nonburnable classes
#   = 45 valid non-NoData numeric codes.
FBFM40_NONBURNABLE_CODES = set(
    FBFM40_GROUP_CODES["NB"]
)

FBFM40_BURNABLE_CODES = (
    FBFM40_VALID_CODES
    - FBFM40_NONBURNABLE_CODES
)

assert len(FBFM40_NONBURNABLE_CODES) == 5
assert len(FBFM40_BURNABLE_CODES) == 40
assert len(FBFM40_VALID_CODES) == 45

print(
    "FBFM40 DOMAIN:",
    f"{len(FBFM40_BURNABLE_CODES)} burnable + "
    f"{len(FBFM40_NONBURNABLE_CODES)} nonburnable = "
    f"{len(FBFM40_VALID_CODES)} valid codes"
)
print("FBFM40 DOMAIN CONTRACT: PASS")

FBFM40_CODE_TO_GROUP = {
    code: group
    for group, codes in FBFM40_GROUP_CODES.items()
    for code in codes
}

fbfm_group_table = pd.DataFrame([
    {
        "group": group,
        "codes": ",".join(
            str(v)
            for v in sorted(codes)
        ),
        "code_count": len(codes),
    }
    for group, codes in FBFM40_GROUP_CODES.items()
])

display(fbfm_group_table)

fbfm_group_table.to_csv(
    OUTPUT_ROOT / "FBFM40_GROUP_MAPPING_V5.csv",
    index=False,
)

FBFM40_NB_SUBCODES = {
    "NB1": 91,
    "NB2": 92,
    "NB3": 93,
    "NB8": 98,
    "NB9": 99,
}

assert set(FBFM40_NB_SUBCODES.values()) == FBFM40_NONBURNABLE_CODES

print("FBFM40 NB SUBCLASS CONTRACT: PASS")


## 7. Weighted aggregation contracts

FBFM40 codes are never averaged.

For each region, the core representation uses:

- group proportions: NB, GR, GS, SH, TU, TL, SB;
- burnable / nonburnable fraction;
- dominant group fraction;
- exact-model entropy;
- group entropy;
- number of unique exact models.

The extended representation additionally retains proportions for all 45 valid non-NoData FBFM40 raster codes.

Canopy products are summarized in physical/nominal units only **after** masking `-9999`.

Zero is retained as a substantive non-forest state.

CC and CH are thematic/class-representative values; their weighted means summarize spatial class composition and are **not** interpreted as subpixel interpolation.

For CH/CBH/CBD, top-bin/censored fractions are retained explicitly.

In [ ]:
def _clean_weighted(values, weights):
    v = np.asarray(values, dtype=float)
    w = np.asarray(weights, dtype=float)

    ok = (
        np.isfinite(v)
        & np.isfinite(w)
        & (w > 0)
    )

    return v[ok], w[ok]


def weighted_mean(values, weights):
    v, w = _clean_weighted(values, weights)

    if not len(v):
        return np.nan

    return float(
        np.average(v, weights=w)
    )


def weighted_std(values, weights):
    v, w = _clean_weighted(values, weights)

    if not len(v):
        return np.nan

    mu = np.average(v, weights=w)

    return float(
        np.sqrt(
            np.average(
                (v - mu) ** 2,
                weights=w,
            )
        )
    )


def weighted_quantile(values, weights, q):
    v, w = _clean_weighted(values, weights)

    if not len(v):
        return np.nan

    order = np.argsort(v)
    v = v[order]
    w = w[order]

    cumulative = np.cumsum(w)
    target = float(q) * cumulative[-1]

    return float(
        v[
            np.searchsorted(
                cumulative,
                target,
                side="left",
            )
        ]
    )


def weighted_fraction(mask, weights):
    mask = np.asarray(mask, dtype=bool)
    w = np.asarray(weights, dtype=float)

    ok = np.isfinite(w) & (w > 0)

    if not ok.any():
        return np.nan

    return float(
        w[ok & mask].sum()
        / w[ok].sum()
    )


def weighted_entropy(categories, weights):
    c = np.asarray(categories)
    w = np.asarray(weights, dtype=float)

    ok = (
        pd.notna(c)
        & np.isfinite(w)
        & (w > 0)
    )

    if not ok.any():
        return np.nan

    c = c[ok]
    w = w[ok]

    totals = {}

    for value, weight in zip(c, w):
        totals[value] = (
            totals.get(value, 0.0)
            + float(weight)
        )

    p = np.asarray(
        list(totals.values()),
        dtype=float,
    )

    p = p / p.sum()

    return float(
        -(p * np.log(p)).sum()
    )


def aggregate_fbfm40(raw_values, weights):
    raw = pd.to_numeric(
        pd.Series(raw_values),
        errors="coerce",
    ).to_numpy(float)

    w = np.asarray(weights, dtype=float)

    valid = (
        np.isfinite(raw)
        & (raw != -9999)
        & np.isin(
            raw.astype(
                int,
                copy=False,
            ),
            list(FBFM40_VALID_CODES),
        )
    )

    out = {
        "valid_weight_fraction": weighted_fraction(
            valid,
            w,
        ),
    }

    if not valid.any():
        for group in FBFM40_GROUP_CODES:
            out[f"group_{group}_fraction"] = np.nan

        out.update({
            "dominant_group_fraction": np.nan,
            "exact_entropy": np.nan,
            "group_entropy": np.nan,
            "unique_exact_models": np.nan,
        })

        for nb_name in FBFM40_NB_SUBCODES:
            out[f"{nb_name.lower()}_fraction"] = np.nan

        for code in sorted(FBFM40_VALID_CODES):
            out[f"exact_code_{code}_fraction"] = np.nan

        return out

    vr = raw[valid].astype(int)
    vw = w[valid]

    groups = np.asarray([
        FBFM40_CODE_TO_GROUP[int(code)]
        for code in vr
    ])

    for group in FBFM40_GROUP_CODES:
        out[f"group_{group}_fraction"] = weighted_fraction(
            groups == group,
            vw,
        )

    # Individual nonburnable subclasses are retained because the
    # five NB classes have different operational interpretations.
    for nb_name, nb_code in FBFM40_NB_SUBCODES.items():
        out[f"{nb_name.lower()}_fraction"] = weighted_fraction(
            vr == nb_code,
            vw,
        )

    group_fracs = [
        out[f"group_{group}_fraction"]
        for group in FBFM40_GROUP_CODES
    ]

    out["dominant_group_fraction"] = float(
        np.nanmax(group_fracs)
    )

    out["exact_entropy"] = weighted_entropy(
        vr,
        vw,
    )

    out["group_entropy"] = weighted_entropy(
        groups,
        vw,
    )

    out["unique_exact_models"] = int(
        len(np.unique(vr))
    )

    for code in sorted(FBFM40_VALID_CODES):
        out[f"exact_code_{code}_fraction"] = weighted_fraction(
            vr == code,
            vw,
        )

    return out


CANOPY_SPEC = {
    "CC": {
        "scale": 1.0,
        "top_censored_raw": None,
        "units": "percent",
        "value_semantics": "thematic_class_midpoint",
    },
    "CH": {
        "scale": 0.1,
        "top_censored_raw": 510,
        "units": "m_nominal",
        "value_semantics": "thematic_class_value_top_bin_right_censored",
    },
    "CBH": {
        "scale": 0.1,
        "top_censored_raw": 100,
        "units": "m_nominal",
        "value_semantics": "continuous_steps_with_top_bin_right_censored",
    },
    "CBD": {
        "scale": 0.01,
        "top_censored_raw": 45,
        "units": "kg_m3_nominal",
        "value_semantics": "continuous_steps_with_top_bin_right_censored",
    },
}


def aggregate_canopy(
    product,
    raw_values,
    weights,
):
    spec = CANOPY_SPEC[product]

    raw = pd.to_numeric(
        pd.Series(raw_values),
        errors="coerce",
    ).to_numpy(
        dtype=float,
        copy=True,
    )

    w = np.asarray(
        weights,
        dtype=float,
    )

    valid = (
        np.isfinite(raw)
        & (raw != -9999)
    )

    out = {
        "valid_weight_fraction": weighted_fraction(
            valid,
            w,
        ),
    }

    if not valid.any():
        for stat in [
            "mean",
            "std",
            "p10",
            "median",
            "p90",
            "zero_fraction",
            "positive_mean",
            "positive_std",
            "positive_p10",
            "positive_median",
            "positive_p90",
            "top_censored_fraction",
        ]:
            out[stat] = np.nan

        return out

    vr = raw[valid]
    vw = w[valid]

    physical = (
        vr * float(spec["scale"])
    )

    out["mean"] = weighted_mean(
        physical,
        vw,
    )

    out["std"] = weighted_std(
        physical,
        vw,
    )

    out["p10"] = weighted_quantile(
        physical,
        vw,
        0.10,
    )

    out["median"] = weighted_quantile(
        physical,
        vw,
        0.50,
    )

    out["p90"] = weighted_quantile(
        physical,
        vw,
        0.90,
    )

    out["zero_fraction"] = weighted_fraction(
        vr == 0,
        vw,
    )

    positive = vr > 0

    if positive.any():
        vp = physical[positive]
        wp = vw[positive]

        out["positive_mean"] = weighted_mean(vp, wp)
        out["positive_std"] = weighted_std(vp, wp)
        out["positive_p10"] = weighted_quantile(vp, wp, 0.10)
        out["positive_median"] = weighted_quantile(vp, wp, 0.50)
        out["positive_p90"] = weighted_quantile(vp, wp, 0.90)
    else:
        out["positive_mean"] = np.nan
        out["positive_std"] = np.nan
        out["positive_p10"] = np.nan
        out["positive_median"] = np.nan
        out["positive_p90"] = np.nan

    top = spec["top_censored_raw"]

    out["top_censored_fraction"] = (
        weighted_fraction(
            vr == top,
            vw,
        )
        if top is not None
        else np.nan
    )

    return out


## 7B. Joint canopy-derived features

CH, CBH and CBD must be sampled on the same native 30 m grid.

After masking `-9999` and scaling to physical units, compute per cell:

- `crown_depth_m = max(CH - CBH, 0)`
- `cfl_kg_m2 = CBD × crown_depth_m`
- `cbh_gt_ch` structural inconsistency indicator.

These are derived **before** regional aggregation.


In [ ]:

def derive_canopy_structure(
    ch_raw,
    cbh_raw,
    cbd_raw,
):
    ch_raw = pd.to_numeric(
        pd.Series(ch_raw),
        errors="coerce",
    ).to_numpy(dtype=float, copy=True)

    cbh_raw = pd.to_numeric(
        pd.Series(cbh_raw),
        errors="coerce",
    ).to_numpy(dtype=float, copy=True)

    cbd_raw = pd.to_numeric(
        pd.Series(cbd_raw),
        errors="coerce",
    ).to_numpy(dtype=float, copy=True)

    valid = (
        np.isfinite(ch_raw)
        & np.isfinite(cbh_raw)
        & np.isfinite(cbd_raw)
        & (ch_raw != -9999)
        & (cbh_raw != -9999)
        & (cbd_raw != -9999)
    )

    ch_m = np.full(len(ch_raw), np.nan, dtype=float)
    cbh_m = np.full(len(ch_raw), np.nan, dtype=float)
    cbd = np.full(len(ch_raw), np.nan, dtype=float)

    ch_m[valid] = ch_raw[valid] / 10.0
    cbh_m[valid] = cbh_raw[valid] / 10.0
    cbd[valid] = cbd_raw[valid] / 100.0

    inconsistent = (
        valid
        & (cbh_m > ch_m)
    )

    crown_depth = np.full(
        len(ch_raw),
        np.nan,
        dtype=float,
    )

    crown_depth[valid] = np.maximum(
        ch_m[valid] - cbh_m[valid],
        0.0,
    )

    cfl = np.full(
        len(ch_raw),
        np.nan,
        dtype=float,
    )

    cfl[valid] = (
        cbd[valid]
        * crown_depth[valid]
    )

    return pd.DataFrame({
        "joint_valid": valid,
        "cbh_gt_ch": inconsistent,
        "crown_depth_m": crown_depth,
        "cfl_kg_m2": cfl,
    })


def aggregate_derived_canopy(
    ch_raw,
    cbh_raw,
    cbd_raw,
    weights,
):
    derived = derive_canopy_structure(
        ch_raw,
        cbh_raw,
        cbd_raw,
    )

    w = np.asarray(weights, dtype=float)
    valid = derived["joint_valid"].to_numpy(bool)

    out = {
        "joint_valid_weight_fraction": weighted_fraction(
            valid,
            w,
        ),
        "cbh_gt_ch_fraction": weighted_fraction(
            derived["cbh_gt_ch"].to_numpy(bool),
            w,
        ),
    }

    if not valid.any():
        for name in [
            "crown_depth_mean",
            "crown_depth_p90",
            "cfl_mean",
            "cfl_p90",
        ]:
            out[name] = np.nan
        return out

    vw = w[valid]
    depth = derived.loc[valid, "crown_depth_m"].to_numpy(float)
    cfl = derived.loc[valid, "cfl_kg_m2"].to_numpy(float)

    out["crown_depth_mean"] = weighted_mean(depth, vw)
    out["crown_depth_p90"] = weighted_quantile(depth, vw, 0.90)
    out["cfl_mean"] = weighted_mean(cfl, vw)
    out["cfl_p90"] = weighted_quantile(cfl, vw, 0.90)

    return out


## 8. Synthetic aggregation QA

In [ ]:
# FBFM40: do not average 101 and 204.
fuel_test = aggregate_fbfm40(
    [101, 101, 204, 91],
    [1, 1, 1, 1],
)

assert math.isclose(
    fuel_test["group_GR_fraction"],
    0.5,
)

assert math.isclose(
    fuel_test["group_SB_fraction"],
    0.25,
)

assert math.isclose(
    fuel_test["group_NB_fraction"],
    0.25,
)

# V4+ intentionally does NOT emit burnable/nonburnable fractions:
#   nonburnable == group_NB_fraction
#   burnable == 1 - group_NB_fraction
# so those fields would be exact duplicates.
assert "burnable_fraction" not in fuel_test
assert "nonburnable_fraction" not in fuel_test

# The single NB cell in this synthetic example is code 91 (NB1).
assert math.isclose(
    fuel_test["nb1_fraction"],
    0.25,
)

# Broad-group composition must form a simplex.
assert math.isclose(
    sum(
        fuel_test[f"group_{group}_fraction"]
        for group in FBFM40_GROUP_CODES
    ),
    1.0,
)

# Canopy zero is VALID, -9999 is missing.
cc_test = aggregate_canopy(
    "CC",
    [0, 15, 95, -9999],
    [1, 1, 1, 1],
)

assert math.isclose(
    cc_test["valid_weight_fraction"],
    0.75,
)

assert math.isclose(
    cc_test["zero_fraction"],
    1/3,
)

# CH top class is retained and flagged as censored.
ch_test = aggregate_canopy(
    "CH",
    [0, 190, 510],
    [1, 1, 1],
)

assert math.isclose(
    ch_test["top_censored_fraction"],
    1/3,
)

print("AGGREGATION CONTRACT QA: PASS")

# Positive/forest-conditional canopy summary excludes zero.
assert math.isclose(
    cc_test["positive_mean"],
    (15 + 95) / 2,
)

# Derived crown structure.
derived_test = aggregate_derived_canopy(
    ch_raw=[200, 100, 0],
    cbh_raw=[50, 120, 0],
    cbd_raw=[20, 10, 0],
    weights=[1, 1, 1],
)

# cell1 depth 15 m, CFL .2*15 = 3 kg/m2
# cell2 is inconsistent and is clamped to depth 0
assert math.isclose(
    derived_test["crown_depth_mean"],
    5.0,
)
assert math.isclose(
    derived_test["cfl_mean"],
    1.0,
)
assert math.isclose(
    derived_test["cbh_gt_ch_fraction"],
    1/3,
)

print("V4 CANOPY-DERIVED QA: PASS")


## 9. Freeze the feature dictionary

In [ ]:

feature_rows = []

REGIONS = list(REGION_DEPTHS_M)

def add_feature(
    feature,
    tier,
    region,
    product,
    statistic,
    units,
    description,
    direction_required=None,
):
    if direction_required is None:
        direction_required = region in DIRECTIONAL_REGIONS

    feature_rows.append({
        "feature": feature,
        "tier": tier,
        "region": region,
        "product": product,
        "statistic": statistic,
        "units": units,
        "categorical": False,
        "direction_required": direction_required,
        "description": description,
    })


# ------------------------------------------------------------
# QA support / unique-cell fields.
# ------------------------------------------------------------
for region in REGIONS:
    for product in PRODUCTS:
        add_feature(
            f"landfire_{region}_{product.lower()}_valid_weight_fraction",
            QA_TIER,
            region,
            product,
            "valid_weight_fraction",
            "fraction",
            "Weighted valid LANDFIRE support fraction.",
        )

        add_feature(
            f"landfire_{region}_{product.lower()}_unique_cell_count",
            QA_TIER,
            region,
            product,
            "unique_cell_count",
            "count",
            "Number of unique native 30 m cells after duplicate-hit collapse.",
        )


# ------------------------------------------------------------
# FBFM40 CORE:
# broad composition + NB subclasses + broad entropy.
# Burnable/nonburnable are deleted because NB and 1-NB duplicate them.
# ------------------------------------------------------------
for region in REGIONS:
    for group in FBFM40_GROUP_CODES:
        add_feature(
            f"landfire_{region}_fbfm_group_{group.lower()}_fraction",
            CORE_TIER,
            region,
            "FBFM40",
            f"group_{group}_fraction",
            "fraction",
            f"Weighted broad FBFM40 group {group} fraction.",
        )

    for nb_name, nb_code in FBFM40_NB_SUBCODES.items():
        add_feature(
            f"landfire_{region}_fbfm_{nb_name.lower()}_fraction",
            CORE_TIER,
            region,
            "FBFM40",
            f"{nb_name.lower()}_fraction",
            "fraction",
            f"Weighted fraction of exact nonburnable code {nb_name} ({nb_code}).",
        )

    add_feature(
        f"landfire_{region}_fbfm_group_entropy",
        CORE_TIER,
        region,
        "FBFM40",
        "group_entropy",
        "nats",
        "Weighted Shannon entropy over broad fuel groups.",
    )

    # Derived/redundant or sample-size-sensitive fields move to extended.
    for stat, units, desc in [
        (
            "exact_entropy",
            "nats",
            "Weighted Shannon entropy over exact FBFM40 classes.",
        ),
        (
            "unique_exact_models",
            "count",
            "Number of represented exact FBFM40 classes; sample-size sensitive.",
        ),
        (
            "dominant_group_fraction",
            "fraction",
            "Maximum broad-group fraction; deterministic from group composition.",
        ),
    ]:
        add_feature(
            f"landfire_{region}_fbfm_{stat}",
            EXTENDED_TIER,
            region,
            "FBFM40",
            stat,
            units,
            desc,
        )

    for code_value in sorted(FBFM40_VALID_CODES):
        add_feature(
            f"landfire_{region}_fbfm_code_{code_value}_fraction",
            EXTENDED_TIER,
            region,
            "FBFM40",
            f"exact_code_{code_value}_fraction",
            "fraction",
            f"Weighted exact FBFM40 code {code_value} fraction.",
        )


# ------------------------------------------------------------
# Canopy.
#
# Core:
#   unconditional mean + zero fraction +
#   positive-conditioned mean/std.
#
# CBH/CBD additionally retain positive p10/median/p90 in core.
# CC/CH discrete percentiles move to extended.
# ------------------------------------------------------------
for region in REGIONS:
    for product, spec in CANOPY_SPEC.items():

        for stat, units, desc in [
            (
                "mean",
                spec["units"],
                "Unconditional landscape mean; valid zero retained.",
            ),
            (
                "zero_fraction",
                "fraction",
                "Fraction of valid support represented by source zero/nonforest.",
            ),
            (
                "positive_mean",
                spec["units"],
                "Mean conditional on positive source value.",
            ),
            (
                "positive_std",
                spec["units"],
                "Standard deviation conditional on positive source value.",
            ),
        ]:
            add_feature(
                f"landfire_{region}_{product.lower()}_{stat}",
                CORE_TIER,
                region,
                product,
                stat,
                units,
                desc,
            )

        if product in {"CBH", "CBD"}:
            for stat in [
                "positive_p10",
                "positive_median",
                "positive_p90",
            ]:
                add_feature(
                    f"landfire_{region}_{product.lower()}_{stat}",
                    CORE_TIER,
                    region,
                    product,
                    stat,
                    spec["units"],
                    f"{stat} conditional on positive source value.",
                )
        else:
            # Discrete/tied class distributions: percentile set is extended.
            for stat in [
                "p10",
                "median",
                "p90",
                "positive_p10",
                "positive_median",
                "positive_p90",
                "std",
            ]:
                add_feature(
                    f"landfire_{region}_{product.lower()}_{stat}",
                    EXTENDED_TIER,
                    region,
                    product,
                    stat,
                    spec["units"],
                    f"Extended distribution summary for {product}.",
                )

        if spec["top_censored_raw"] is not None:
            add_feature(
                f"landfire_{region}_{product.lower()}_top_censored_fraction",
                CORE_TIER,
                region,
                product,
                "top_censored_fraction",
                "fraction",
                "Fraction in the documented top source class; interpretation follows product metadata.",
            )


# ------------------------------------------------------------
# Joint canopy-derived structure.
# ------------------------------------------------------------
for region in REGIONS:
    for stat, units, desc in [
        (
            "crown_depth_mean",
            "m",
            "Weighted mean max(CH-CBH,0) on jointly valid cells.",
        ),
        (
            "crown_depth_p90",
            "m",
            "Weighted p90 max(CH-CBH,0) on jointly valid cells.",
        ),
        (
            "cfl_mean",
            "kg_m2",
            "Weighted mean CBD*max(CH-CBH,0) on jointly valid cells.",
        ),
        (
            "cfl_p90",
            "kg_m2",
            "Weighted p90 CBD*max(CH-CBH,0) on jointly valid cells.",
        ),
        (
            "cbh_gt_ch_fraction",
            "fraction",
            "Fraction of jointly valid cells where CBH exceeds CH before clamping.",
        ),
    ]:
        add_feature(
            f"landfire_{region}_{stat}",
            CORE_TIER,
            region,
            "DERIVED_CANOPY",
            stat,
            units,
            desc,
        )

    add_feature(
        f"landfire_{region}_derived_canopy_joint_valid_weight_fraction",
        QA_TIER,
        region,
        "DERIVED_CANOPY",
        "joint_valid_weight_fraction",
        "fraction",
        "Joint CH/CBH/CBD valid support fraction.",
    )


# ------------------------------------------------------------
# Fireward-protected contrasts.
#
# Signed differences are fireward - protected.
# JSD is symmetric and nonnegative.
# ------------------------------------------------------------
for scale in ["near", "far"]:
    add_feature(
        f"landfire_{scale}_fbfm_group_js_divergence",
        CORE_TIER,
        f"{scale}_contrast",
        "FBFM40",
        "jensen_shannon_divergence",
        "nats",
        "Jensen-Shannon divergence between fireward and protected broad FBFM40 compositions.",
        direction_required=True,
    )

    for quantity, units in [
        ("cc_positive_mean", "percent"),
        ("cbd_positive_mean", "kg_m3_nominal"),
        ("crown_depth_mean", "m"),
        ("cfl_mean", "kg_m2"),
        ("fbfm_nb8_fraction", "fraction"),
        ("fbfm_nb9_fraction", "fraction"),
    ]:
        add_feature(
            f"landfire_{scale}_fireward_minus_protected_{quantity}",
            CORE_TIER,
            f"{scale}_contrast",
            "CONTRAST",
            f"fireward_minus_protected_{quantity}",
            units,
            "Signed contrast: fireward region minus protected region.",
            direction_required=True,
        )


# ------------------------------------------------------------
# Weakest-point / positional upper-tail summaries.
# These are produced before native-cell region pooling, retaining line position.
# ------------------------------------------------------------
for scale in ["near", "far"]:
    for quantity, units in [
        ("cbd_positive_mean", "kg_m3_nominal"),
        ("cfl_mean", "kg_m2"),
    ]:
        add_feature(
            f"landfire_fireward_{scale}_position_p90_{quantity}",
            CORE_TIER,
            f"fireward_{scale}",
            "POSITIONAL",
            f"position_p90_{quantity}",
            units,
            "Arc-length-weighted p90 across line positions of per-position fireward quantity.",
            direction_required=True,
        )

        add_feature(
            f"landfire_fireward_{scale}_position_max_{quantity}",
            EXTENDED_TIER,
            f"fireward_{scale}",
            "POSITIONAL",
            f"position_max_{quantity}",
            units,
            "Raw maximum across line positions; extended because it is outlier-sensitive.",
            direction_required=True,
        )


# ------------------------------------------------------------
# Provenance / missingness.
# ------------------------------------------------------------
for feature, description in [
    (
        "landfire_version_key",
        "Frozen retrospective LANDFIRE vintage key; provenance only.",
    ),
    (
        "landfire_source_status",
        "Resolved/archive-pending source state; provenance only.",
    ),
    (
        "landfire_directional_context_valid",
        "Frozen 04A direction reliability / directional-feature availability flag.",
    ),
]:
    feature_rows.append({
        "feature": feature,
        "tier": QA_TIER,
        "region": "unit",
        "product": "provenance",
        "statistic": "metadata",
        "units": "none",
        "categorical": True,
        "direction_required": False,
        "description": description,
    })


feature_dictionary = (
    pd.DataFrame(feature_rows)
    .drop_duplicates("feature")
    .sort_values(
        ["tier", "region", "product", "feature"]
    )
    .reset_index(drop=True)
)

tier_counts = (
    feature_dictionary.groupby(
        ["tier", "product"],
        as_index=False,
    )
    .agg(features=("feature", "size"))
)

display(tier_counts)

print("Total frozen fields:", f"{len(feature_dictionary):,}")
print(
    "Core model features:",
    f"{(feature_dictionary['tier'] == CORE_TIER).sum():,}",
)
print(
    "Extended model features:",
    f"{(feature_dictionary['tier'] == EXTENDED_TIER).sum():,}",
)
print(
    "QA-only fields:",
    f"{(feature_dictionary['tier'] == QA_TIER).sum():,}",
)

feature_dictionary.to_csv(
    OUTPUT_ROOT / "LANDFIRE_FEATURE_DICTIONARY_V5.csv",
    index=False,
)



## 10. Feature-tier rationale

### Core

The core representation now prioritizes nonredundant, physically interpretable
features:

- broad FBFM40 group composition;
- individual NB subclasses;
- broad-group entropy;
- unconditional plus positive-conditioned canopy structure;
- derived crown depth and canopy fuel load;
- explicit fireward-versus-protected contrasts;
- robust p90 weakest-point summaries.

The exact duplicate `burnable_fraction` / `nonburnable_fraction` pair is removed
because it is algebraically determined by the NB proportion.

### Extended

Extended features contain:

- all 45 exact FBFM40 class proportions;
- exact-class entropy;
- unique exact-model count;
- dominant-group fraction;
- discrete CC/CH percentile summaries;
- raw positional maxima.

These remain available for ablation without burdening the first model.

### QA-only

Vintage/source status, directional availability, valid-support fractions and
unique native-cell counts are retained for provenance and data-quality audits.

They are not automatically used as predictors.



## 10B. Contrast and weakest-point aggregation contracts

These are production contracts for 04D1D.

### Broad-group Jensen–Shannon divergence

For normalized broad-group distributions \(P\) and \(Q\),

\[
JS(P,Q)
=
\frac12 KL(P\|M)
+
\frac12 KL(Q\|M),
\qquad
M=\frac12(P+Q).
\]

### Positional p90

Before region-level duplicate-cell pooling, retain each originating line
position. For selected fireward quantities, aggregate the offset-depth samples
within position, then compute an arc-length-weighted p90 across positions.

This represents a robust upper-tail / weakest-section exposure rather than a
single outlier maximum.


In [ ]:

def jensen_shannon_divergence(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)

    if (
        len(p) != len(q)
        or len(p) == 0
        or not np.isfinite(p).all()
        or not np.isfinite(q).all()
        or p.sum() <= 0
        or q.sum() <= 0
    ):
        return np.nan

    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)

    def kl(a, b):
        mask = a > 0
        return float(
            np.sum(
                a[mask]
                * np.log(
                    a[mask] / b[mask]
                )
            )
        )

    return 0.5 * kl(p, m) + 0.5 * kl(q, m)


# Symmetry / identity QA.
assert math.isclose(
    jensen_shannon_divergence(
        [0.5, 0.5],
        [0.5, 0.5],
    ),
    0.0,
)

_js_a = jensen_shannon_divergence(
    [1.0, 0.0],
    [0.0, 1.0],
)

_js_b = jensen_shannon_divergence(
    [0.0, 1.0],
    [1.0, 0.0],
)

assert math.isclose(_js_a, _js_b)
assert _js_a > 0

print("CONTRAST / JSD CONTRACT: PASS")


def weighted_position_p90(
    position_values,
    position_weights,
):
    """
    Robust upper-tail summary across line positions.

    position_values:
        one already-aggregated environmental quantity per line position.
    position_weights:
        original arc-length weight represented by each line position.
    """
    return weighted_quantile(
        position_values,
        position_weights,
        0.90,
    )


## 11. Quantify existing 03L line-sample workload

In [ ]:
sample_count_rows = []

block_files = sorted(
    TERRAIN_BLOCK_SAMPLE_ROOT.glob("*.parquet")
)

print("Terrain block sample shards:", f"{len(block_files):,}")

for path in block_files:
    # Fast metadata row count.
    metadata = pq.ParquetFile(path).metadata

    sample_count_rows.append({
        "terrain_block_id": path.stem,
        "line_sample_rows": metadata.num_rows,
    })

sample_block_counts = pd.DataFrame(
    sample_count_rows
)

display(
    sample_block_counts[
        "line_sample_rows"
    ].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    ).to_frame().T
)

print(
    "Total 03L line sample rows:",
    f"{sample_block_counts['line_sample_rows'].sum():,}",
)

## 12. Production workload estimate

For a reliable-direction unit, each 03L sample generates:

- 1 line position;
- 6 fireward positions;
- 6 protected positions;

for **13 total context positions**.

For an unreliable-direction unit, only the line position is generated.

This estimate counts spatial sampling locations, not REST calls.

04D1D will perform local array indexing after downloading **one raster tile per product × vintage × 10 km block**, rather than issuing one web request per sample.

In [ ]:
# Need per-unit line sample counts, so read only unit_uid from block shards.
unit_sample_frames = []

for path in block_files:
    x = pd.read_parquet(
        path,
        columns=["unit_uid"],
    )

    count = (
        x.groupby(
            "unit_uid",
            as_index=False,
        )
        .size()
        .rename(
            columns={
                "size": "line_sample_count"
            }
        )
    )

    unit_sample_frames.append(count)

unit_sample_counts = pd.concat(
    unit_sample_frames,
    ignore_index=True,
)

# A unit belongs to one terrain block, so one row/unit is expected.
if unit_sample_counts["unit_uid"].duplicated().any():
    unit_sample_counts = (
        unit_sample_counts.groupby(
            "unit_uid",
            as_index=False,
        )["line_sample_count"]
        .sum()
    )

assert unit_sample_counts["unit_uid"].is_unique

work = base[
    [
        "unit_uid",
        "landfire_version_key",
        "landfire_source_status",
        "terrain_block_id",
        "landfire_directional_context_valid",
    ]
].merge(
    unit_sample_counts,
    on="unit_uid",
    how="left",
    validate="one_to_one",
)

if work["line_sample_count"].isna().any():
    raise RuntimeError(
        "Some operational units have no 03L/terrain line samples."
    )

work["directional_depth_count"] = np.where(
    work["landfire_directional_context_valid"],
    sum(
        len(REGION_DEPTHS_M[r])
        for r in DIRECTIONAL_REGIONS
    ),
    0,
)

work["requested_spatial_positions"] = (
    work["line_sample_count"]
    * (
        1
        + work["directional_depth_count"]
    )
)

workload_by_vintage = (
    work.groupby(
        [
            "landfire_version_key",
            "landfire_source_status",
        ],
        as_index=False,
    )
    .agg(
        units=("unit_uid", "size"),
        line_samples=("line_sample_count", "sum"),
        requested_spatial_positions=(
            "requested_spatial_positions",
            "sum",
        ),
        blocks=("terrain_block_id", "nunique"),
    )
)

workload_by_vintage[
    "five_product_array_lookups"
] = (
    workload_by_vintage[
        "requested_spatial_positions"
    ]
    * len(PRODUCTS)
)

display(workload_by_vintage)

workload_by_vintage.to_csv(
    OUTPUT_ROOT / "LANDFIRE_PRODUCTION_WORKLOAD_ESTIMATE_V5.csv",
    index=False,
)

## 13. Freeze the 10 km tile production manifest

In [ ]:
# One tile set per vintage × spatial block.
tile_manifest = (
    base.groupby(
        [
            "landfire_version_key",
            "landfire_source_status",
            "terrain_block_id",
            "terrain_block_ix",
            "terrain_block_iy",
        ],
        as_index=False,
    )
    .agg(
        units=("unit_uid", "size"),
        incidents=("_irwin_norm", "nunique"),
        directional_valid_units=(
            "landfire_directional_context_valid",
            "sum",
        ),
    )
)

tile_manifest["core_minx"] = (
    tile_manifest["terrain_block_ix"]
    * BLOCK_SIZE_M
)

tile_manifest["core_miny"] = (
    tile_manifest["terrain_block_iy"]
    * BLOCK_SIZE_M
)

tile_manifest["core_maxx"] = (
    tile_manifest["core_minx"]
    + BLOCK_SIZE_M
)

tile_manifest["core_maxy"] = (
    tile_manifest["core_miny"]
    + BLOCK_SIZE_M
)

tile_manifest["requested_minx"] = (
    tile_manifest["core_minx"]
    - LANDFIRE_BLOCK_HALO_M
)

tile_manifest["requested_miny"] = (
    tile_manifest["core_miny"]
    - LANDFIRE_BLOCK_HALO_M
)

tile_manifest["requested_maxx"] = (
    tile_manifest["core_maxx"]
    + LANDFIRE_BLOCK_HALO_M
)

tile_manifest["requested_maxy"] = (
    tile_manifest["core_maxy"]
    + LANDFIRE_BLOCK_HALO_M
)

tile_manifest["product_tile_count"] = len(PRODUCTS)

tile_manifest["production_eligible_now"] = (
    tile_manifest["landfire_source_status"]
    == "resolved"
)

# Rough native tile size before exact service-grid snapping.
tile_width_m = (
    BLOCK_SIZE_M
    + 2 * LANDFIRE_BLOCK_HALO_M
)

approx_pixels_per_axis = math.ceil(
    tile_width_m / 30.0
)

approx_pixels_per_tile = (
    approx_pixels_per_axis ** 2
)

approx_s16_mb = (
    approx_pixels_per_tile
    * 2
    / (1024**2)
)

tile_manifest[
    "approx_uncompressed_s16_mb_per_product"
] = approx_s16_mb

tile_manifest[
    "approx_uncompressed_s16_mb_all_5_products"
] = (
    approx_s16_mb
    * len(PRODUCTS)
)

display(
    tile_manifest.groupby(
        [
            "landfire_version_key",
            "landfire_source_status",
        ],
        as_index=False,
    )
    .agg(
        version_blocks=("terrain_block_id", "size"),
        units=("units", "sum"),
        product_tiles=("product_tile_count", "sum"),
        approx_uncompressed_gb=(
            "approx_uncompressed_s16_mb_all_5_products",
            lambda s: s.sum() / 1024.0,
        ),
    )
)

tile_manifest.to_csv(
    OUTPUT_ROOT / "LANDFIRE_TILE_MANIFEST_V5.csv",
    index=False,
)

## 14. Frozen 04D1D extraction architecture

For every **LANDFIRE vintage × 10 km terrain block**:

1. Load the operational units in the block.
2. Load the existing weighted 03L line samples.
3. Construct line + fireward/protected context positions using the frozen region specification.
4. For each of the five LANDFIRE products:
   - resolve the vintage-specific validated raster source;
   - use that service/file's own native extent and 30 m grid origin;
   - snap the requested 10 km + 360 m halo bbox to native cell edges;
   - retrieve/cache one raw S16 nearest-neighbor raster tile;
   - validate CRS, 30 m pixel size, S16, NoData and product domain;
   - index every requested point locally using the returned raster transform;
   - record native raster row/column for each hit;
   - collapse duplicate native cells within unit × region and sum their arc-length weights.
5. Join CH/CBH/CBD on native cell identity and derive crown depth, CFL, and canopy inconsistency.
6. Aggregate native-cell values by unit × region using the frozen weighted functions.
7. Compute fireward-vs-protected contrasts.
8. Compute robust positional p90 fireward exposure summaries before discarding line-position identity.
9. Force directional-region features to missing when `landfire_directional_context_valid=False`.
10. Write one restartable Parquet feature shard per vintage × block.
11. Record a run-log row with product/domain/coverage QA.
12. Concatenate only after all expected shards for the requested vintage set pass.

### Critical performance rule

**No per-unit REST sampling in production.**

The validated REST APIs are source mechanisms, not the unit-level computational architecture.

One block tile should serve thousands of local array lookups.

## 15. Missingness and leakage rules

In [ ]:
rules = pd.DataFrame([
    {
        "rule": "pre_event_vintage_only",
        "frozen": True,
        "detail": (
            "Use 04D0 frozen retrospective version mapping; "
            "do not substitute newer same-year disturbance products."
        ),
    },
    {
        "rule": "no_label_conditioned_sampling",
        "frozen": True,
        "detail": (
            "Spatial regions are functions only of 03L geometry and "
            "pre-challenge 04A approach direction."
        ),
    },
    {
        "rule": "directional_missing_when_unreliable",
        "frozen": True,
        "detail": (
            "All fireward/protected LANDFIRE features are NaN when the "
            "frozen approach direction is unreliable."
        ),
    },
    {
        "rule": "line_features_remain_available",
        "frozen": True,
        "detail": (
            "Underlying pre-event line-location LANDFIRE features do not "
            "require a reliable approach direction."
        ),
    },
    {
        "rule": "mask_nodata_before_scaling",
        "frozen": True,
        "detail": (
            "-9999 is missing; zero remains a valid non-forest state "
            "for CC/CH/CBH/CBD."
        ),
    },
    {
        "rule": "no_numeric_fbfm_code_averages",
        "frozen": True,
        "detail": (
            "FBFM40 is represented by categorical proportions, "
            "entropy and counts only."
        ),
    },
    {
        "rule": "nearest_neighbor_raw_sampling",
        "frozen": True,
        "detail": (
            "All raw products are sampled at native 30 m with "
            "nearest-neighbor; aggregation occurs only after extraction."
        ),
    },
    {
        "rule": "source_metadata_not_default_predictors",
        "frozen": True,
        "detail": (
            "Version/source-status/coverage QA fields are provenance "
            "fields unless explicitly tested in a separate ablation."
        ),
    },
    {
        "rule": "collapse_duplicate_native_cells_with_summed_length_weight",
        "frozen": True,
        "detail": (
            "After raster indexing, repeated unit-region native cells are "
            "collapsed and their represented arc-length weights are summed."
        ),
    },
    {
        "rule": "approach_direction_strictly_pre_challenge_dependency",
        "frozen": True,
        "detail": (
            "Directional geometry depends on the frozen 04A approach estimate "
            "and must not be recomputed from post-engagement or final-perimeter-fit data."
        ),
    },
    {
        "rule": "direction_availability_is_QA_not_default_predictor",
        "frozen": True,
        "detail": (
            "Direction availability is audited against outcome and retained as QA; "
            "directional-feature sensitivity must be reported."
        ),
    },
])

display(rules)

rules.to_csv(
    OUTPUT_ROOT / "LANDFIRE_FEATURE_RULES_V5.csv",
    index=False,
)

## 16. Freeze machine-readable specification

In [ ]:
spec = {
    "build_id": NOTEBOOK_BUILD_ID,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "version_by_fire_year": VERSION_BY_FIRE_YEAR,
    "source_status": SOURCE_STATUS,
    "products": PRODUCTS,
    "region_depths_m": REGION_DEPTHS_M,
    "directional_regions": sorted(DIRECTIONAL_REGIONS),
    "max_context_offset_m": MAX_CONTEXT_OFFSET_M,
    "block_size_m": BLOCK_SIZE_M,
    "landfire_block_halo_m": LANDFIRE_BLOCK_HALO_M,
    "fbfm40_groups": {
        group: sorted(codes)
        for group, codes in FBFM40_GROUP_CODES.items()
    },
    "fbfm40_valid_code_count": len(FBFM40_VALID_CODES),
    "fbfm40_burnable_code_count": len(FBFM40_BURNABLE_CODES),
    "fbfm40_nonburnable_code_count": len(FBFM40_NONBURNABLE_CODES),
    "fbfm40_nb_subcodes": FBFM40_NB_SUBCODES,
    "native_cell_duplicate_policy": (
        "collapse unit×region×row×col and sum context_weight_m"
    ),
    "contrast_scales": ["near", "far"],
    "derived_canopy": {
        "crown_depth_m": "max(CH_m-CBH_m,0)",
        "cfl_kg_m2": "CBD_kg_m3*crown_depth_m",
        "inconsistency": "CBH_m>CH_m",
    },
    "positional_core_summary": "arc_length_weighted_p90",
    "canopy_spec": CANOPY_SPEC,
    "core_feature_count": int(
        (feature_dictionary["tier"] == CORE_TIER).sum()
    ),
    "extended_feature_count": int(
        (feature_dictionary["tier"] == EXTENDED_TIER).sum()
    ),
    "qa_feature_count": int(
        (feature_dictionary["tier"] == QA_TIER).sum()
    ),
}

SPEC_PATH = (
    PROCESSED_ROOT / "landfire_feature_spec_v5.json"
)

SPEC_PATH.write_text(
    json.dumps(
        spec,
        indent=2,
    ),
    encoding="utf-8",
)

print("Saved:")
print(SPEC_PATH)

## 17. Final architecture checks

In [ ]:
checks = pd.DataFrame([
    {
        "check": "operational_unit_index_unique",
        "pass": bool(base["unit_uid"].is_unique),
    },
    {
        "check": "all_years_have_frozen_version",
        "pass": bool(base["landfire_version_key"].notna().all()),
    },
    {
        "check": "all_versions_have_source_status",
        "pass": bool(base["landfire_source_status"].notna().all()),
    },
    {
        "check": "directional_geometry_contract_pass",
        "pass": True,
    },
    {
        "check": "fbfm40_exact_domain_has_45_codes",
        "pass": len(FBFM40_VALID_CODES) == 45,
    },
    {
        "check": "fbfm40_burnable_domain_has_40_codes",
        "pass": len(FBFM40_BURNABLE_CODES) == 40,
    },
    {
        "check": "fbfm40_nonburnable_domain_has_5_codes",
        "pass": len(FBFM40_NONBURNABLE_CODES) == 5,
    },
    {
        "check": "aggregation_contract_QA_pass",
        "pass": True,
    },
    {
        "check": "native_cell_collapse_contract_pass",
        "pass": True,
    },
    {
        "check": "contrast_js_contract_pass",
        "pass": True,
    },
    {
        "check": "derived_canopy_contract_pass",
        "pass": True,
    },
    {
        "check": "every_operational_unit_has_line_samples",
        "pass": bool(work["line_sample_count"].notna().all()),
    },
    {
        "check": "tile_manifest_covers_all_units",
        "pass": (
            int(tile_manifest["units"].sum())
            == len(base)
        ),
    },
    {
        "check": "resolved_and_pending_sources_explicit",
        "pass": (
            set(base["landfire_source_status"])
            <= {"resolved", "archive_pending"}
        ),
    },
])

display(checks)

if checks["pass"].all():
    print("04D1C LANDFIRE FEATURE ARCHITECTURE: PASS / READY TO FREEZE")
else:
    print("04D1C LANDFIRE FEATURE ARCHITECTURE: REVIEW REQUIRED")

checks.to_csv(
    OUTPUT_ROOT / "LANDFIRE_FEATURE_ARCHITECTURE_CHECKS_V5.csv",
    index=False,
)

## 18. Architecture report

In [ ]:
def tomd(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)

tier_summary = (
    feature_dictionary.groupby(
        ["tier", "product"],
        as_index=False,
    )
    .agg(
        features=("feature", "size")
    )
)

tile_summary = (
    tile_manifest.groupby(
        [
            "landfire_version_key",
            "landfire_source_status",
        ],
        as_index=False,
    )
    .agg(
        blocks=("terrain_block_id", "size"),
        units=("units", "sum"),
        product_tiles=("product_tile_count", "sum"),
    )
)

report = f"""# 04D1C LANDFIRE Feature Architecture

Build: `{NOTEBOOK_BUILD_ID}`

Generated UTC: {datetime.now(timezone.utc).isoformat()}

## Decision

This notebook freezes the LANDFIRE feature architecture before blockwise production.

## Operational universe

Units: {len(base):,}

Directional context valid: {int(base['landfire_directional_context_valid'].sum()):,}
({base['landfire_directional_context_valid'].mean()*100:.3f}%)

## Frozen regions

{tomd(region_spec)}

## Feature tiers

{tomd(tier_summary)}

## Vintage/source status

{tomd(vintage_summary)}

## Production workload

{tomd(workload_by_vintage)}

## Tile architecture

{tomd(tile_summary)}

Native tile architecture:
10 km core blocks with {LANDFIRE_BLOCK_HALO_M:g} m halo.

No per-unit REST sampling is permitted in full production.

## Important semantics

- LANDFIRE values at the line describe the **pre-event underlying landscape**, not the constructed line treatment itself.
- Negative approach-axis offsets are fireward.
- Positive approach-axis offsets are beyond/protected-side.
- Directional regions are missing by design when 04A direction is unreliable.
- FBFM40 class numbers are never averaged; duplicate native-cell hits are collapsed with summed line-length weight.
- CC/CH thematic values may be spatially summarized after extraction but are not interpolated.
- Canopy summaries separate structural zero from positive-condition structure; CH/CBH/CBD top source classes remain explicitly flagged.
- Zero is retained as valid non-forest for canopy products.
- -9999 is masked before scaling.
- Vintage/source metadata is QA/provenance, not a default predictor.

## Architecture checks

{tomd(checks)}

## Next

04D1D should implement a **resolved-vintage production pilot** using this exact machine-readable feature specification and the 10 km tile cache architecture.

LF2019L and LF2020 remain archive-acquisition dependencies; their absence does not reopen the frozen feature design.
"""

REPORT_PATH = (
    OUTPUT_ROOT / "LANDFIRE_FEATURE_ARCHITECTURE_REPORT_V5.md"
)

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

print(report)
print()
print("Saved:", REPORT_PATH)

## 19. Output inventory

In [ ]:
print("=" * 100)
print("04D1C OUTPUTS")
print("=" * 100)

for p in sorted(OUTPUT_ROOT.glob("*")):
    if p.is_file():
        print(
            f"{p.name:<90}"
            f"{p.stat().st_size/(1024**2):>12.3f} MB"
        )

print()
print("Processed specification:")
print(SPEC_PATH)
print()
print("DONE:", NOTEBOOK_BUILD_ID)